# RATING PARTECIPANTI

Questo codice consente di fare la classifica dei soggetti


In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# PATH
# ============================================================

base_path = Path(r"C:\Users\a.genua\OneDrive - Scuola Superiore Sant'Anna\PROGETTI\Progetti in corso\Articoli in corso\RA-L_&_Sanseverino\FlightSimulator")

input_csv = base_path / "stats_subject_condition.csv"

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(input_csv)

print(df.head())

# ============================================================
# CLASSIFICA SOGGETTI
# ============================================================

ranking = (
    df.groupby("subject")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        total_trials=("n_trials", "sum"),
        mean_omission_rate=("omission_rate", "mean")
    )
    .reset_index()
)

# ordina dal migliore al peggiore
ranking = ranking.sort_values(
    by="mean_accuracy",
    ascending=False
).reset_index(drop=True)

# aggiungi posizione in classifica
ranking.insert(0, "rank", ranking.index + 1)

print(ranking)

  subject condition  n_trials  accuracy  omission_rate
0     ID1      HIGH        26  0.653846       0.000000
1     ID1       LOW        26  0.846154       0.000000
2     ID1   NOMINAL        27  0.962963       0.000000
3    ID10      HIGH        26  0.653846       0.000000
4    ID10       LOW        28  0.535714       0.035714
    rank subject  mean_accuracy  std_accuracy  total_trials  \
0      1    ID22       1.000000      0.000000            87   
1      2    ID13       1.000000      0.000000            82   
2      3    ID25       1.000000      0.000000            88   
3      4    ID14       0.988506      0.019909            85   
4      5    ID11       0.988095      0.020620            84   
5      6    ID18       0.988095      0.020620            85   
6      7    ID27       0.977778      0.038490            89   
7      8    ID12       0.976984      0.019968            86   
8      9    ID24       0.953968      0.039936            89   
9     10    ID23       0.952778      0.0

In [2]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# PATH
# ============================================================

base_path = Path(r"C:\Users\a.genua\OneDrive - Scuola Superiore Sant'Anna\PROGETTI\Progetti in corso\Articoli in corso\RA-L_&_Sanseverino\FlightSimulator")

ranking_csv = base_path / "stats_subject_condition.csv"
excel_path = base_path / "2-BACK-TASK-analisi_V2.xlsx"

# ============================================================
# LOAD DATI STATISTICI
# ============================================================

df = pd.read_csv(ranking_csv)

# ============================================================
# RECUPERO NOMI DA EXCEL
# ============================================================

df_excel = pd.read_excel(excel_path, header=None)

# riga 0: soggetti, con celle merge
header_subjects = df_excel.iloc[0, :].ffill()

id_name_map = {}

for cell in header_subjects:
    if pd.isna(cell):
        continue

    text = str(cell).strip()

    # cerca pattern tipo: 1 (Cristian Camardella)
    match = re.search(r"(\d+)\s*\((.*?)\)", text)

    if match:
        subj_id = f"ID{int(match.group(1))}"
        name = match.group(2).strip()
        id_name_map[subj_id] = name

print("Mappa ID -> Nome:")
for k, v in id_name_map.items():
    print(k, "->", v)

# ============================================================
# CLASSIFICA CON NOMI
# ============================================================

ranking = (
    df.groupby("subject")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        total_trials=("n_trials", "sum"),
        mean_omission_rate=("omission_rate", "mean")
    )
    .reset_index()
)

ranking["name"] = ranking["subject"].map(id_name_map)

ranking = ranking.sort_values(
    by=["mean_accuracy", "mean_omission_rate", "total_trials"],
    ascending=[False, True, False]
).reset_index(drop=True)

ranking.insert(0, "rank", ranking.index + 1)

ranking_display = ranking.copy()
ranking_display["mean_accuracy_%"] = (ranking_display["mean_accuracy"] * 100).round(2)
ranking_display["mean_omission_rate_%"] = (ranking_display["mean_omission_rate"] * 100).round(2)

ranking_display = ranking_display[
    [
        "rank",
        "subject",
        "name",
        "mean_accuracy_%",
        "std_accuracy",
        "total_trials",
        "mean_omission_rate_%"
    ]
]

print("\nCLASSIFICA FINALE:")
print(ranking_display)

# ============================================================
# TOP 3
# ============================================================

top3 = ranking_display.head(3)

print("\n🏆 TOP 3:")
print(top3[["rank", "subject", "name", "mean_accuracy_%"]])

# ============================================================
# SALVA CSV
# ============================================================

output_csv = base_path / "ranking_subjects_2back_accuracy_with_names.csv"
ranking_display.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"\nClassifica salvata in:\n{output_csv}")

Mappa ID -> Nome:
ID1 -> Cristian Camardella
ID2 -> Alessandro Genua
ID3 -> Federica Serra
ID4 -> Andrea Losacco
ID5 -> Camilla Celli
ID6 -> Rui Chan
ID7 -> Lorenzo Sterzi
ID8 -> Eleonora Lanfranco
ID9 -> Valerio Novelli
ID10 -> Gabriele Nerucci
ID11 -> Giorgio
ID12 -> Giovanni
ID13 -> Anastasios Tzepkenlis
ID14 -> Sveva Quinzii
ID15 -> Pia
ID16 -> Edoardo Gaspari
ID17 -> Matilda
ID18 -> Pietro
ID19 -> Andrea Soldato
ID20 -> Giacomo
ID21 -> Pietro Mammini
ID22 -> Miriam Gentile
ID23 -> Sabrina Di Giuseppe
ID24 -> Simone Scala
ID25 -> Matteo Caponi
ID26 -> Lorenzo Della Rosa
ID27 -> Alessio Giuliani
ID28 -> Anna Baldisseri
ID29 -> Pietro Vassallo
ID30 -> Angelo

CLASSIFICA FINALE:
    rank subject                   name  mean_accuracy_%  std_accuracy  \
0      1    ID25          Matteo Caponi           100.00      0.000000   
1      2    ID22         Miriam Gentile           100.00      0.000000   
2      3    ID13  Anastasios Tzepkenlis           100.00      0.000000   
3      4    ID1

In [3]:
# ============================================================
# ESCLUDI ORGANIZZATORI DALLA CLASSIFICA
# ============================================================

organizers = [
    "Cristian Camardella",
    "Anastasios Tzepkenlis",
    "Alessandro Genua",
    "Federica Serra",
]

ranking_no_organizers = ranking_display[
    ~ranking_display["name"].isin(organizers)
].copy()

# ricalcola il rank dopo l'esclusione
ranking_no_organizers = ranking_no_organizers.reset_index(drop=True)
ranking_no_organizers["rank"] = ranking_no_organizers.index + 1

print("CLASSIFICA SENZA ORGANIZZATORI:")
print(ranking_no_organizers)

# ============================================================
# TOP 3 SENZA ORGANIZZATORI
# ============================================================

top3_no_organizers = ranking_no_organizers.head(3)

print("\n🏆 TOP 3 SENZA ORGANIZZATORI:")
for _, row in top3_no_organizers.iterrows():
    print(
        f"{int(row['rank'])}° posto: "
        f"{row['name']} ({row['subject']}) - "
        f"{row['mean_accuracy_%']}%"
    )

# ============================================================
# SALVA CSV
# ============================================================

output_csv_no_org = base_path / "ranking_subjects_2back_accuracy_no_organizers.csv"

ranking_no_organizers.to_csv(
    output_csv_no_org,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nClassifica senza organizzatori salvata in:\n{output_csv_no_org}")


CLASSIFICA SENZA ORGANIZZATORI:
    rank subject                 name  mean_accuracy_%  std_accuracy  \
0      1    ID25        Matteo Caponi           100.00      0.000000   
1      2    ID22       Miriam Gentile           100.00      0.000000   
2      3    ID14        Sveva Quinzii            98.85      0.019909   
3      4    ID18               Pietro            98.81      0.020620   
4      5    ID11              Giorgio            98.81      0.020620   
5      6    ID27     Alessio Giuliani            97.78      0.038490   
6      7    ID12             Giovanni            97.70      0.019968   
7      8    ID24         Simone Scala            95.40      0.039936   
8      9    ID23  Sabrina Di Giuseppe            95.28      0.017347   
9     10    ID20              Giacomo            94.51      0.066625   
10    11    ID19       Andrea Soldato            92.94      0.062570   
11    12    ID16      Edoardo Gaspari            84.52      0.268055   
12    13    ID17              Ma